

# **Descarga de licitaciones en plazo del País Vasco: API rest de la Plataforma KontratazioA**

Notebook editado el 04/09/2026

---
## **Set-up**
---


Instalación e importación de librerías:

In [1]:
!pip install pandas sodapy sentence-transformers supabase

In [9]:
from datetime import datetime, date,  timedelta
from google.colab import userdata
import os
import time
from sodapy import Socrata
import pandas as pd
from sentence_transformers import SentenceTransformer
from supabase import create_client, Client
import requests

Configuración de la Base de Datos:

In [3]:
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

Carga del modelo de embbedings:

In [4]:
print("Cargando modelo de IA (multilingual-e5-small)...")
encoder = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")

Cargando modelo de IA (multilingual-e5-small)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

---
## **Funciones**
---

Añade País Vasco al lugar de ejecución detectado.

In [5]:
MAPEO_NUTS_EUSKADI = {
    "ES211": "Álava/Araba",
    "ES212": "Gipuzkoa",
    "ES213": "Bizkaia",
    "CAPV": "País Vasco"
}

def procesar_lugar_euskadi(lugar_raw):
    """Añade País Vasco al lugar de ejecución detectado."""
    lugar_limpio = str(lugar_raw).strip() if lugar_raw else "No especificado"
    if lugar_limpio == "No especificado" or not lugar_limpio:
        return "País Vasco"

    lugar_lower = lugar_limpio.lower()
    if "país vasco" not in lugar_lower and "euskadi" not in lugar_lower:
        return f"{lugar_limpio}, País Vasco"
    return lugar_limpio

In [6]:
def normalizar_organo(org):
    """Extrae la raíz del órgano eliminando subcategorías tras guiones o sufijos."""
    if not org:
        return ""
    org_limpio = org.split("-")[0].split("—")[0].strip().lower()
    return org_limpio

---
## **Extracción y sincronización**
---

In [10]:
def sincronizar_licitaciones_euskadi():
    hoy = datetime.now().date()
    base_url = "https://api.euskadi.eus/administration/events"
    headers = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}

    tipo_licitacion_id = "1"  # Contrataciones administrativas

    fecha_actual = datetime(2026, 9, 1).date()
    fecha_fin_rango = datetime(2026, 9, 7).date()

    results = []
    print(f"Consultando la API de Euskadi por fechas del {fecha_actual} al {fecha_fin_rango}...")

    while fecha_actual <= fecha_fin_rango:
        year = fecha_actual.year
        month = fecha_actual.month
        day = fecha_actual.day

        url = f"{base_url}/v1.0/events/byType/{tipo_licitacion_id}/byDate/{year}/{month}/{day}"
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 200:
                data = response.json()
                events = data.get("events", []) if isinstance(data, dict) else data
                for ev in events:
                    results.append((fecha_actual, ev))
        except Exception as e:
            print(f"Error conectando con la API de Euskadi para la fecha {fecha_actual}: {e}")

        fecha_actual += timedelta(days=1)

    print(f"Descargados {len(results)} registros totales de la API de Euskadi.\n")

    try:
        existentes_resp = supabase.table("licitaciones").select("enlace, titulo, organo").execute()
        mapa_enlaces = {item["enlace"] for item in existentes_resp.data if "enlace" in item}

        registros_existentes = set()
        for item in existentes_resp.data:
            t = str(item.get("titulo", "")).strip().lower()
            o_base = normalizar_organo(item.get("organo", ""))
            if t or o_base:
                registros_existentes.add((t, o_base))

        print(f"Registros cargados desde Supabase para validación: {len(registros_existentes)}")
    except Exception as e:
        print(f"Error conectando con Supabase: {e}")
        return

    licitaciones_validas = []
    filtrados_caducados = 0
    filtrados_duplicados = 0
    enlaces_ya_procesados_en_sesion = set()
    claves_sesion = set()

    for fecha_ev, aviso in results:
        enlace = aviso.get("urlEs", "")
        codigo_item = aviso.get("record", "")
        if not enlace:
            enlace = f"https://www.contratacion.euskadi.eus/webkpe00-kpeperfi/es/contenidos/anuncio_contratacion/{codigo_item}/es_doc/index.html"

        titulo_str = str(
            aviso.get("nameEs") or
            aviso.get("nameEu") or
            "Sin título"
        ).strip()

        organo_raw = str(aviso.get("adjudicatorEs", "No especificado")).strip()
        organo_str = organo_raw
        organo_base = normalizar_organo(organo_raw)

        fecha_fin_str = "No especificada"
        deadline_raw = aviso.get("endDate")
        if deadline_raw:
            fecha_fin_str = deadline_raw[:10]
            try:
                cierre_date = datetime.strptime(fecha_fin_str, "%Y-%m-%d").date()
                if cierre_date < hoy:
                    filtrados_caducados += 1
                    continue
            except ValueError:
                pass

        fecha_pub = str(aviso.get("startDate", ""))[:10]

        # Extracción y depuración de campos para encontrar el importe y CPV correctos
        importe = 0.0
        cpv = "No especificado"

        if codigo_item:
            try:
                url_detalle = f"https://api.euskadi.eus/procurements/contracting-notices/{codigo_item}"
                resp_detalle = requests.get(url_detalle, headers={"Accept": "application/json"}, timeout=5)
                if resp_detalle.status_code == 200:
                    det_data = resp_detalle.json()

                    # IMPRINT DE PRUEBA: Muestra las claves y estructura del JSON de detalle para depurar
                    print(f"\n--- PRUEBA DETALLE [Código: {codigo_item}] ---")
                    print("Claves principales en det_data:", list(det_data.keys()))
                    if "contractingAuthority" in det_data:
                        print("Claves en contractingAuthority:", list(det_data["contractingAuthority"].keys()))
                    print("Presupuestos encontrados:", {
                        "budgetWithoutVAT": det_data.get("budgetWithoutVAT"),
                        "estimatedValue": det_data.get("estimatedValue"),
                        "budgetWithVAT": det_data.get("budgetWithVAT"),
                        "budget": det_data.get("budget")
                    })
                    print("CPV encontrado:", det_data.get("CPV") or det_data.get("contractingAuthority", {}).get("CPV"))
                    print("--------------------------------------------------\n")

                    importe = float(
                        det_data.get("budgetWithoutVAT") or
                        det_data.get("estimatedValue") or
                        det_data.get("budgetWithVAT") or
                        det_data.get("budget") or 0.0
                    )

                    cpv_raw = det_data.get("CPV") or det_data.get("contractingAuthority", {}).get("CPV", "No especificado")
                    if isinstance(cpv_raw, list):
                        cpv_nombres = [c.get("name", "") for c in cpv_raw if isinstance(c, dict) and c.get("name")]
                        cpv = ", ".join(cpv_nombres) if cpv_nombres else str(cpv_raw)
                    elif isinstance(cpv_raw, dict):
                        cpv = cpv_raw.get("name", str(cpv_raw))
                    else:
                        cpv = str(cpv_raw)
            except Exception as ex:
                print(f"Error consultando detalle para {codigo_item}: {ex}")

        lugar_ejecucion = procesar_lugar_euskadi("País Vasco")

        clave_duplicado = (titulo_str.lower(), organo_base)
        if (enlace in mapa_enlaces or
            enlace in enlaces_ya_procesados_en_sesion or
            clave_duplicado in registros_existentes or
            clave_duplicado in claves_sesion):
            filtrados_duplicados += 1
            continue

        enlaces_ya_procesados_en_sesion.add(enlace)
        claves_sesion.add(clave_duplicado)

        texto_completo = f"passage: Título: {titulo_str}. Órgano: {organo_str}. CPV: {cpv}. Lugar: {lugar_ejecucion}. Importe: {importe} EUR."
        embedding = encoder.encode(texto_completo).tolist()

        elemento = {
            "titulo": titulo_str,
            "organo": organo_str,
            "fecha": fecha_pub,
            "importe": importe,
            "enlace": enlace,
            "texto_completo": texto_completo,
            "embedding": embedding,
            "fecha_fin": fecha_fin_str,
            "lugar_ejecucion": lugar_ejecucion,
            "cpv": cpv,
            "es_novedad": True,
            "es_actualizada": False,
            "fuente": "Euskadi"
        }

        licitaciones_validas.append(elemento)

    print(f"\n--- ESTADÍSTICAS EUSKADI ---")
    print(f"Descartados por fecha caducada: {filtrados_caducados}")
    print(f"Duplicados evitados (con órgano normalizado): {filtrados_duplicados}")
    print(f"Licitaciones válidas listas para insertar: {len(licitaciones_validas)}\n")

    if licitaciones_validas:
        print("Subiendo licitaciones de Euskadi a Supabase...")
        tamano_lote = 15
        max_intentos = 3

        for i in range(0, len(licitaciones_validas), tamano_lote):
            lote = licitaciones_validas[i:i + tamano_lote]
            num_lote = i // tamano_lote + 1
            exito = False

            for intento in range(1, max_intentos + 1):
                try:
                    supabase.table("licitaciones").upsert(lote, on_conflict="enlace").execute()
                    print(f"  -> Lote Euskadi {num_lote} procesado con éxito ({len(lote)} registros).")
                    exito = True
                    break
                except Exception as e:
                    print(f"Intento {intento}/{max_intentos} fallido para lote Euskadi {num_lote}: {e}")
                    if intento < max_intentos:
                        time.sleep(2 * intento)
                    else:
                        print(f"Error definitivo al subir lote Euskadi {num_lote}.")

        print("¡Sincronización de Euskadi completada con éxito!")
    else:
        print("No hay nuevas licitaciones de Euskadi para insertar.")

if __name__ == "__main__":
    sincronizar_licitaciones_euskadi()

Consultando la API de Euskadi por fechas del 2026-09-01 al 2026-09-07...
Descargados 81 registros totales de la API de Euskadi.

Registros cargados desde Supabase para validación: 8076

--- ESTADÍSTICAS EUSKADI ---
Descartados por fecha caducada: 0
Duplicados evitados (con órgano normalizado): 57
Licitaciones válidas listas para insertar: 24

Subiendo licitaciones de Euskadi a Supabase...
  -> Lote Euskadi 1 procesado con éxito (15 registros).
  -> Lote Euskadi 2 procesado con éxito (9 registros).
¡Sincronización de Euskadi completada con éxito!
